# NutriRecall — Prototype Notebook

## 1. Imports and project paths

In [20]:
import os
from pathlib import Path
from datetime import datetime
from getpass import getpass
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

file_path = PROJECT_ROOT / "data" / "health_logs.csv"
if not file_path.is_file():
    raise FileNotFoundError(f"Expected existing health log: {file_path}")

## 2. Choose one model backend

`USE_LOCAL = True` → Ollama · `False` → Groq API.

In [37]:
USE_LOCAL = False  # True = Ollama, False = Groq API
LOCAL_MODEL = "phi3:mini"
API_MODEL = "openai/gpt-oss-20b"
RUN_MODEL_TEST = True
RUN_ASSISTANT = True
#ENABLE_LEGACY_MEMORY = False

## 3. Optional: append one health log

Set `ADD_ENTRY = True` to save a new entry. Re-running can append duplicates.

In [38]:
ADD_ENTRY = False

if ADD_ENTRY:
    date = datetime.today().strftime("%Y-%m-%d")
    weight = float(input("Enter weight (kg): "))
    protein = float(input("Protein intake (g): "))
    sleep = float(input("Sleep hours: "))
    workout = int(input("Workout? (1=yes, 0=no): "))

    entry = {
        "date": date, "weight": weight, "protein": protein,
        "sleep_hours": sleep, "workout": workout,
    }
    # Respect the existing header order and spelling when appending.
    headers = pd.read_csv(file_path, nrows=0).columns.tolist()
    normalized = [name.strip().lower() for name in headers]
    if len(normalized) != len(entry) or set(normalized) != set(entry):
        raise ValueError(f"Unexpected CSV columns: {headers}")
    pd.DataFrame(
        [[entry[name] for name in normalized]], columns=headers
    ).to_csv(file_path, mode="a", header=False, index=False)
    ADD_ENTRY = False
    print("Entry saved. Reload the data below.")


## 4. Load and normalize the CSV

Normalize column names and legacy dates.

In [39]:
data = pd.read_csv(file_path)

# Match CSV headers to the names used in the notebook.
data.columns = data.columns.str.strip()
data = data.rename(columns={
    "date": "Date",
    "weight": "Weight",
    "protein": "Protein",
    "sleep_hours": "Sleep_hours",
    "workout": "Workout",
})

# Fix existing two-digit years, such as 26-09-29.
dates = data["Date"].astype("string").str.strip()
dates = dates.str.replace(
    r"^(\d{2})-(\d{2})-(\d{2})$",
    r"20\1-\2-\3",
    regex=True,
)

data["Date"] = pd.to_datetime(dates, format="ISO8601", errors="raise")
data = data.sort_values("Date").reset_index(drop=True)

data.head()

,Date,Weight,Protein,Sleep_hours,Workout
0,2026-04-22 18:01:34.872205,56.4,20.0,7.0,1
1,2026-04-24 00:00:00.000000,55.4,15.0,7.0,1
2,2026-09-29 00:00:00.000000,77.0,70.0,5.5,0
3,2026-09-29 00:00:00.000000,70.0,90.0,6.0,0


## 5. Prototype health calculations

Protein, sleep, workout, and weighted health scores.

In [40]:
data ['Protein_required'] = data['Weight'] * 1.6
data ['Protein_score'] = data['Protein']/ data['Protein_required']
data ['sleep_score'] = data['Sleep_hours']/7
data ['workout_score'] = data['Workout']

data['Protein_score'] = data['Protein_score'].clip(0,1)
data['sleep_score'] = data['sleep_score'].clip(0,1)

data['health_score'] =(
    0.4 *data['Protein_score'] +
    0.3 * data['sleep_score'] +
    0.3 * data['workout_score']
)

data['health_score_10'] = (data['health_score'] *9) +1

recent = data.tail(7)

avg_health = recent['health_score'].mean()
avg_protein = recent['Protein'].mean()
avg_sleep = recent['Sleep_hours'].mean()
workout_days = recent['Workout'].sum()

## 6. Patterns and summary

Uses the latest seven records, not necessarily seven calendar days.

In [41]:
def detect_patterns(data):
    recent = data.tail(7)
    issues = []

    if recent['Protein'].mean() < recent['Protein_required'].mean():
        issues.append("Consistently low protein intake")

    if recent['Sleep_hours'].mean() < 6:
        issues.append("Poor sleep pattern")

    if recent['Workout'].sum() < 3:
        issues.append("Low workout consistency")

    if recent['health_score'].mean() < 0.6:
        issues.append("Overall lifestyle needs improvement")

    return issues

def generate_insights(data):
    recent = data.tail(7)
    issues = detect_patterns(data)

    insight = f"""
    Last 7 Days Summary:
    Avg Protein: {recent['Protein'].mean():.1f}g
    Avg sleep: {recent['Sleep_hours'].mean():.1f}hrs
    Workout Days: {recent['Workout'].sum()}/7
    Health score: {recent['health_score_10'].mean():.1f} / 10

    Key Issues:
    {", ".join(issues) if issues else "No Major Issues"}
    """
    return insight

In [42]:
print(generate_insights(data))


    Last 7 Days Summary:
    Avg Protein: 48.8g
    Avg sleep: 6.4hrs
    Workout Days: 2/7
    Health score: 6.4 / 10

    Key Issues:
    Consistently low protein intake, Low workout consistency, Overall lifestyle needs improvement
    


## 7. Existing placeholder knowledge and TF-IDF retrieval

Placeholder knowledge; retrieve the closest statement.

In [43]:
knowledge = [
    "Protein intake for muscle growth should be 1.6 to 2.2 grams per kg body weight, and Optimal per meal: 0.3 – 0.5 g/kg",
    "Sleeping less than 6 hours reduces recovery and muscle growth.",
    "Consistent workouts are required to build muscle.",
    "Low protein intake can prevent muscle gain.",
    "High-quality sources: Eggs, chicken, fish Dairy (curd, paneer, milk) Plant: lentils, chickpeas, tofu",
    "Ideal sleep: 7–9 hours/night",
    "IF protein < 1.6 g/kg → suggest increasing intake",
    "IF sleep < 6 hrs → warn about recovery drop",
    "Recovery is as important as training for muscle growth."
]

In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics.pairwise import cosine_similarity

vectorize = TfidfVectorizer()
X = vectorize.fit_transform(knowledge)

def retrieve_knowledge(query):
    query_vec = vectorize.transform([query])
    similarity = cosine_similarity(query_vec, X)

    index = similarity.argmax()
    return knowledge[index]

In [45]:
print(retrieve_knowledge("muscle growth"))

Sleeping less than 6 hours reduces recovery and muscle growth.


## 8. Optional legacy Chroma memory demonstration

Disabled by default. Legacy experiment; excluded from the minor scope.

In [46]:
collection = None
if ENABLE_LEGACY_MEMORY:
    import chromadb
    chroma_client = chromadb.EphemeralClient()
    collection = chroma_client.get_or_create_collection(name="NutriRecall_Memory")

def store_memory(insight, memory_id):
    if not ENABLE_LEGACY_MEMORY or collection is None:
        return
    collection.upsert(documents=[insight], ids=[str(memory_id)])

def retrieve_memory(query):
    if not ENABLE_LEGACY_MEMORY or collection is None:
        return ""
    count = collection.count()
    if count == 0:
        return ""
    results = collection.query(query_texts=[query], n_results=min(2, count))
    return "\n".join(results["documents"][0])

if ENABLE_LEGACY_MEMORY:
    store_memory(generate_insights(data), memory_id="latest_insight")


## 9. One model response function

Local Ollama or Groq API via `USE_LOCAL`. API key comes from the environment or a private prompt.

In [47]:
groq_client = None

def generate_response(prompt):
    global groq_client
    messages = [{"role": "user", "content": prompt}]

    if USE_LOCAL:
        import ollama
        response = ollama.chat(model=LOCAL_MODEL, messages=messages)
        return response["message"]["content"]

    else:
        from openai import OpenAI
        if groq_client is None:
            api_key = os.getenv("GROQ_API_KEY") or getpass("Groq API key: ")
            if not api_key.strip():
                raise ValueError("A Groq API key is required for API mode.")
            groq_client = OpenAI(
                api_key=api_key.strip(),
                base_url="https://api.groq.com/openai/v1",
            )
        response = groq_client.chat.completions.create(
            model=API_MODEL, messages=messages
        )
        return response.choices[0].message.content


### Optional connection test

Enable `RUN_MODEL_TEST` to test the selected model.

In [48]:
if RUN_MODEL_TEST:
    print(generate_response("Say hello in one sentence."))
else:
    print("Model connection test skipped.")

Hello!


## 10. Personalized assistant

Health summary + retrieved knowledge → response.

In [49]:
def nutrirecall_ai(query, data):
    insight = generate_insights(data)
    knowledge_piece = retrieve_knowledge(query)
    memory = retrieve_memory(query)
    memory_section = f"\nPast Memory:\n{memory}\n" if memory else ""

    prompt = f"""You are a fitness and nutrition assistant.

User data:
{insight}

Relevant knowledge:
{knowledge_piece}
{memory_section}
User question:
{query}

Give a short, clear, personalized, practical answer in bullet points.
"""
    return generate_response(prompt)


### Ask a question using the selected backend

Enable `RUN_ASSISTANT`. API mode sends the question and health summary to Groq.

In [50]:
QUESTION = "Why am I not gaining muscle?"

if RUN_ASSISTANT:
    print(nutrirecall_ai(QUESTION, data))
else:
    print("Assistant request skipped. Set RUN_ASSISTANT = True to run.")

- **Protein is the building block** – at ~49 g/day you’re below the 1.2‑1.6 g/kg target needed for muscle growth.  
- **Workouts are too sparse** – only 2 days/week isn’t enough stimulus for hypertrophy.  
- **Sleep is short** – 6.4 h limits recovery and hormone balance (testosterone, growth hormone).  
- **Nutrition quality matters** – if most calories come from carbs or fats, protein gets “diluted.”  

**What to do next**  
- Add 2‑3 protein‑rich snacks (Greek yogurt, whey shake, nuts) to hit 1.5 g/kg.  
- Schedule 3‑4 strength sessions per week; focus on compound lifts (squat, deadlift, bench).  
- Aim for 7–8 h sleep; set a consistent bedtime routine.  
- Track macros in an app—ensure 30–35% calories from protein.  

By aligning protein, training, and sleep, your body will have the fuel and recovery to build muscle.
